# Git for Machine Learning Projects

## Overview
Standard git workflows break down for ML projects: models are binary blobs, datasets are gigabytes, and experiments need parameter tracking beyond code. This notebook covers strategies and tools specifically for ML version control.

---

## 1. Challenges of Git for ML

| Challenge | Standard Git Problem | Solution |
|-----------|---------------------|----------|
| Large model files | git repo becomes huge | Git LFS, DVC |
| Large datasets | can't store in git | DVC, cloud storage |
| Notebook diffs | JSON diffs are unreadable | nbstripout, reviewNB |
| Experiment tracking | commits don't capture hyperparams | MLflow, W&B |
| Reproducibility | `requirements.txt` isn't enough | Docker, conda lock files |

---

## 2. Git LFS (Large File Storage)

Git LFS replaces large files with text pointers in git, storing actual content on a separate server.

```bash
# Install git-lfs
git lfs install

# Track file types
git lfs track "*.pkl"
git lfs track "*.pt"
git lfs track "*.h5"
git lfs track "*.onnx"
git lfs track "*.safetensors"
git lfs track "data/*.csv"

# This creates/updates .gitattributes
git add .gitattributes
git commit -m "chore: configure git lfs tracking"

# Now add large files normally
git add model.pkl
git commit -m "feat: add trained RF model v1"
git push  # LFS files pushed to LFS server

# Check what's tracked
git lfs ls-files
git lfs status
```

**Limitations**: GitHub LFS free tier is 1GB. Use DVC for larger files.

---

## 3. DVC (Data Version Control)

DVC adds data and model versioning on top of git, with any remote storage backend.

### Concepts
- `.dvc` files: lightweight git-tracked pointers (like `.gitignore` sidecars)
- DVC remote: S3, GCS, Azure, SSH, local path
- DVC pipelines: DAG of stages with inputs/outputs tracked

```bash
# Setup
pip install dvc dvc-s3  # or dvc-gs, dvc-azure
dvc init

# Configure remote storage
dvc remote add -d myremote s3://my-ml-bucket/dvc

# Track a dataset
dvc add data/train.csv
git add data/train.csv.dvc data/.gitignore
git commit -m "data: add training dataset v1"
dvc push  # Upload to S3

# Track a model
dvc add models/random_forest.pkl
git add models/random_forest.pkl.dvc
git commit -m "model: trained RF model, accuracy=0.91"
dvc push

# Reproduce on another machine
git clone <repo> && dvc pull
```

### DVC Pipelines
```yaml
# dvc.yaml
stages:
  preprocess:
    cmd: python src/preprocess.py
    deps: [data/raw.csv, src/preprocess.py]
    outs: [data/processed.csv]

  train:
    cmd: python src/train.py --lr ${lr}
    deps: [data/processed.csv, src/train.py]
    params: [params.yaml:lr, params.yaml:n_estimators]
    outs: [models/model.pkl]
    metrics: [metrics.json]

  evaluate:
    cmd: python src/evaluate.py
    deps: [models/model.pkl, data/test.csv]
    metrics: [metrics.json]
```

```bash
dvc repro          # Run pipeline (skips unchanged stages)
dvc dag            # Visualize pipeline DAG
dvc metrics show   # Show metrics
dvc params diff    # Compare parameters
```

---

## 4. Experiment Tracking with Git Tags

Simple model versioning without external tools:
```bash
# Tag a model version
git tag -a model-v1.2.0 -m "RF model: accuracy=0.91, f1=0.88, lr=0.1"
git push origin model-v1.2.0

# List all model versions
git tag -l "model-*"

# Checkout a specific model version
git checkout model-v1.1.0
```

---

## 5. Jupyter Notebook Best Practices

### Problem: Git diffs for notebooks are unreadable JSON

**Solution 1: nbstripout** strip outputs before committing
```bash
pip install nbstripout
nbstripout --install  # Install as git filter
```

**Solution 2: jupytext** sync notebooks with .py scripts
```bash
pip install jupytext
jupytext --to py:percent notebook.ipynb  # Convert to Python script
# Commit the .py file, ignore the .ipynb
```

**Solution 3: nbdime** notebook-aware diff and merge
```bash
pip install nbdime
nbdime config-git --enable  # Enable for git diff/merge
nbdiff notebook_v1.ipynb notebook_v2.ipynb
```

---

## 6. MLflow + Git Integration

MLflow automatically logs the git commit hash with each run:
```python
import mlflow
import subprocess

with mlflow.start_run():
    # Log git info
    git_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
    git_branch = subprocess.check_output(['git', 'branch', '--show-current']).decode().strip()
    mlflow.set_tag('git.commit', git_hash)
    mlflow.set_tag('git.branch', git_branch)
    
    # Log params & metrics
    mlflow.log_params({'lr': 0.01, 'epochs': 10})
    mlflow.log_metric('accuracy', 0.91)
```

---

## 7. Semantic Versioning for Models

Follow SemVer for models: `MAJOR.MINOR.PATCH`

| Change | Version Bump | Example |
|--------|-------------|--------|
| Incompatible API change | MAJOR | v1→v2: different input schema |
| New capability, backward compatible | MINOR | v1.0→v1.1: new features added |
| Bug fix / retrain | PATCH | v1.0.0→v1.0.1: fixed data bug |

---

## 8. GitHub Actions for ML CI/CD

```yaml
# .github/workflows/ml-ci.yml
name: ML CI
on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install -r requirements.txt
      - run: pytest tests/ -v
      - run: python src/train.py --test-mode  # Quick training run
      - name: Check metrics threshold
        run: python scripts/check_metrics.py --min-accuracy 0.85
```

In [1]:
import subprocess

# ── Git helper for ML experiments ──────────────────────────────
class MLGitTracker:
    """Track git metadata for ML experiments."""
    
    @staticmethod
    def _run(cmd):
        try:
            return subprocess.check_output(cmd.split(), stderr=subprocess.DEVNULL).decode().strip()
        except Exception:
            return 'unknown'
    
    @classmethod
    def get_info(cls):
        return {
            'commit': cls._run('git rev-parse HEAD'),
            'short_commit': cls._run('git rev-parse --short HEAD'),
            'branch': cls._run('git branch --show-current'),
            'tag': cls._run('git describe --tags --abbrev=0'),
            'is_dirty': cls._run('git status --porcelain') != '',
            'remote': cls._run('git config --get remote.origin.url'),
        }
    
    @classmethod
    def tag_experiment(cls, metrics: dict, model_version: str):
        """Create a git tag for an experiment."""
        tag_name = f"exp/{model_version}"
        msg = ' '.join(f"{k}={v:.4f}" for k, v in metrics.items())
        subprocess.run(['git', 'tag', '-a', tag_name, '-m', msg])
        return tag_name

info = MLGitTracker.get_info()
print("Git tracking info for current experiment:")
for k, v in info.items():
    print(f"  {k}: {v}")

Git tracking info for current experiment:
  commit: unknown
  short_commit: unknown
  branch: unknown
  tag: unknown
  is_dirty: True
  remote: unknown


In [2]:
# ── .gitignore generator for ML projects ───────────────────────
def generate_ml_gitignore() -> str:
    return """# === Python ===
__pycache__/
*.py[cod]
*$py.class
.venv/
venv/
env/
*.egg-info/
dist/
build/

# === Jupyter ===
.ipynb_checkpoints/
*.ipynb_checkpoints

# === Data (use DVC instead) ===
data/raw/
data/processed/
data/external/
*.csv
*.parquet
*.feather
*.h5
*.hdf5

# === Models (use DVC/LFS) ===
models/
*.pkl
*.joblib
*.pt
*.pth
*.ckpt
*.onnx
*.safetensors
*.bin

# === Secrets ===
.env
.env.*
*.pem
*.key
credentials.json
service_account.json

# === MLOps tools ===
mlruns/
wandb/
.dvc/cache/

# === IDE ===
.vscode/
.idea/
*.swp

# === OS ===
.DS_Store
Thumbs.db
"""

print(generate_ml_gitignore())

# === Python ===
__pycache__/
*.py[cod]
*$py.class
.venv/
venv/
env/
*.egg-info/
dist/
build/

# === Jupyter ===
.ipynb_checkpoints/
*.ipynb_checkpoints

# === Data (use DVC instead) ===
data/raw/
data/processed/
data/external/
*.csv
*.parquet
*.feather
*.h5
*.hdf5

# === Models (use DVC/LFS) ===
models/
*.pkl
*.joblib
*.pt
*.pth
*.ckpt
*.onnx
*.safetensors
*.bin

# === Secrets ===
.env
.env.*
*.pem
*.key
credentials.json
service_account.json

# === MLOps tools ===
mlruns/
wandb/
.dvc/cache/

# === IDE ===
.vscode/
.idea/
*.swp

# === OS ===
.DS_Store
Thumbs.db



## Additional Learning Resources

### Tools
- [Git LFS Docs](https://git-lfs.com/)
- [DVC Documentation](https://dvc.org/doc)
- [nbstripout](https://github.com/kynan/nbstripout)
- [nbdime Notebook diff & merge](https://nbdime.readthedocs.io/)
- [jupytext](https://jupytext.readthedocs.io/)

### Articles
- [DVC Tutorial Iterative.ai](https://iterative.ai/blog/dvc-3-announcement)
- [Git for Data Science Towards Data Science](https://towardsdatascience.com/git-for-data-science)
- [MLOps with DVC and CML](https://cml.dev/doc/start/)

### GitHub
- [DVC Examples](https://github.com/iterative/example-get-started)
- [CML Examples](https://github.com/iterative/cml)